# RAG Pipeline — ร้านฟ้าใหม่ (FahMai)
In-Memory Retrieval-Augmented Generation for Multiple Choice QA

---
**Pipeline Overview:**
1. Ingest `.md` knowledge base files into RAM
2. Vectorize documents with `BAAI/bge-m3`
3. Load `questions.csv` (id, question, choice_1…choice_10)
4. Cosine-similarity retrieval + trace logging
5. LLM generation via Ollama API
6. Fill `sample_submission.csv` template (id, ans) and save to `submission/`

## Cell 1 — Configuration Constants

In [62]:
# ============================================================
#  Configuration Constants
# ============================================================

KNOWLEDGE_BASE_DIR     = "../data/knowledge_base"
QUESTIONS_CSV_PATH     = "../data/questions.csv"
SAMPLE_SUBMISSION_PATH = "../data/sample_submission.csv"
SUBMISSION_DIR         = "../submission"
TRACE_LOG_PATH         = "./trace/trace_log.jsonl"

OLLAMA_API_URL         = "http://localhost:11434/api/generate"
LLM_MODEL_NAME         = "hf.co/mradermacher/typhoon-s-thaillm-8b-instruct-research-preview-GGUF:typhoon-s-thaillm-8b-instruct-research-preview.Q4_K_S.gguf"
EMBEDDING_MODEL_NAME   = "BAAI/bge-m3"
TOP_K_RETRIEVAL        = 3

print("✅ Constants loaded.")

✅ Constants loaded.


## Cell 2 — Imports & Directory Setup

In [2]:
import os
import json
import re
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

Path(TRACE_LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(SUBMISSION_DIR).mkdir(parents=True, exist_ok=True)

print("✅ Imports done. Directories ready.")

/home/coder/project/SuperAi-SS6-MiniHack3/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports done. Directories ready.


## Cell 3 — Step 1: In-Memory Data Ingestion

In [3]:
# ============================================================
#  Step 1 — Load all .md files from knowledge base into RAM
# ============================================================

documents = []  # List[{"filename": str, "content": str}]

for root, dirs, files in os.walk(KNOWLEDGE_BASE_DIR):
    for fname in sorted(files):
        if fname.endswith(".md"):
            filepath = os.path.join(root, fname)
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read().strip()
                documents.append({
                    "filename": os.path.relpath(filepath, KNOWLEDGE_BASE_DIR),
                    "content":  content
                })
            except Exception as e:
                print(f"⚠️  Could not read {filepath}: {e}")

print(f"✅ Loaded {len(documents)} documents into memory.")
for doc in documents:
    print(f"   • {doc['filename']} ({len(doc['content'])} chars)")

✅ Loaded 118 documents into memory.
   • policies/cancellation_policy.md (5159 chars)
   • policies/membership_points_policy.md (7144 chars)
   • policies/return_policy.md (6811 chars)
   • policies/shipping_policy.md (5312 chars)
   • policies/warranty_policy.md (8449 chars)
   • products/AW-MN-001_arcwave_proview_27_4k.md (3273 chars)
   • products/AW-SK-001_arcwave_soundpillar_300.md (2388 chars)
   • products/DN-DT-001_daonuea_tower_x10.md (3015 chars)
   • products/DN-DT-002_daonuea_tower_x10_max.md (3088 chars)
   • products/DN-DT-003_daonuea_mini_pc_m1.md (3509 chars)
   • products/DN-DT-004_daonuea_all_in_one_27.md (3300 chars)
   • products/DN-DT-005_daonuea_all_in_one_24.md (2425 chars)
   • products/DN-LT-001_daonuea_airbook_15.md (4106 chars)
   • products/DN-LT-002_daonuea_airbook_14.md (4402 chars)
   • products/DN-LT-003_daonuea_airbook_14_8gb.md (3987 chars)
   • products/DN-LT-004_daonuea_probook_16.md (5789 chars)
   • products/DN-LT-005_daonuea_probook_14.md (4985 ch

## Cell 4 — Step 2: Vectorization (No Database)

In [ ]:
# ============================================================
#  Step 2 — Embed all documents into a numpy matrix in RAM
# ============================================================

print(f"🔄 Loading embedding model: {EMBEDDING_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

doc_texts = [doc["content"] for doc in documents]

print(f"🔄 Encoding {len(doc_texts)} documents...")
doc_embeddings = embed_model.encode(
    doc_texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True  # unit-norm → cosine sim == dot product
)

print(f"✅ Document matrix shape: {doc_embeddings.shape}")

## Cell 5 — Step 3: Load Questions & Submission Template

In [10]:
# ============================================================
#  Step 3 — Load questions.csv and sample_submission.csv
# ============================================================

# questions.csv: [id, question, choice_1 ... choice_10]
df_questions = pd.read_csv(QUESTIONS_CSV_PATH)
CHOICE_COLS  = [f"choice_{i}" for i in range(1, 11)]

missing_q = [c for c in ["id", "question"] + CHOICE_COLS if c not in df_questions.columns]
assert not missing_q, f"Missing columns in questions.csv: {missing_q}"
print(f"✅ Questions loaded: {len(df_questions)} rows")
display(df_questions.head(3))

# sample_submission.csv: [id, answer] — used as the output template
df_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
assert "id" in df_submission.columns and "answer" in df_submission.columns, \
    "sample_submission.csv must have columns: id, answer"
print(f"\n✅ Sample submission template loaded: {len(df_submission)} rows")
display(df_submission.head(3))

✅ Questions loaded: 100 rows


,id,question,choice_1,choice_2,choice_3,choice_4,choice_5,choice_6,choice_7,choice_8,choice_9,choice_10
0,1,Watch S3 Ultra กันน้ำได้กี่ ATM ครับ,3 ATM,IP68,5 ATM,IP67,10 ATM,20 ATM,IPX8,1 ATM,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
1,2,ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ,"2,990 บาท","4,490 บาท","1,990 บาท","4,990 บาท","3,490 บาท","2,490 บาท","3,990 บาท","5,490 บาท",ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
2,3,หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ,Bluetooth 5.0,Bluetooth 5.3,Bluetooth 5.4,Bluetooth 4.2,Bluetooth 5.2,Bluetooth 5.1,Bluetooth 4.0,Bluetooth 5.5,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า



✅ Sample submission template loaded: 100 rows


,id,answer
0,1,1
1,2,1
2,3,1


In [17]:
df_questions.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         100 non-null    int64
 1   question   100 non-null    str  
 2   choice_1   100 non-null    str  
 3   choice_2   100 non-null    str  
 4   choice_3   100 non-null    str  
 5   choice_4   100 non-null    str  
 6   choice_5   100 non-null    str  
 7   choice_6   100 non-null    str  
 8   choice_7   100 non-null    str  
 9   choice_8   100 non-null    str  
 10  choice_9   100 non-null    str  
 11  choice_10  100 non-null    str  
dtypes: int64(1), str(11)
memory usage: 9.5 KB


## Cell 6 — Helper Functions

In [65]:
# ============================================================
#  Helper Functions (Typhoon-S Optimized 🌪️)
# ============================================================

def retrieve_top_k(question: str, max_k: int = 4, threshold: float = 0.5) -> list[dict]:
    """
    ดึงเอกสารที่เกี่ยวข้อง โดยจำกัดจำนวนไฟล์เพื่อป้องกัน Context ล้น 
    (บน 5090 เราเน้นความแม่นยำของไฟล์ที่คัดมาแล้ว)
    """
    q_vec  = embed_model.encode([question], normalize_embeddings=True)
    scores = cosine_similarity(q_vec, doc_embeddings)[0]
    
    top_indices = np.argsort(scores)[::-1][:max_k]
    results = []
    for i in top_indices:
        score = float(scores[i])
        if score >= threshold:
            results.append({
                "filename": documents[i]["filename"],
                "content":  documents[i].get("content", ""),
                "score":    score
            })
            
    if len(results) == 0:
        best_idx = top_indices[0]
        results.append({
            "filename": documents[best_idx]["filename"],
            "content":  documents[best_idx].get("content", ""),
            "score":    float(scores[best_idx])
        })
    return results

def build_prompt(question: str, choices: list[str], retrieved_docs: list[dict]) -> str:
    """
    สร้าง Prompt สำหรับ Typhoon-S (Instruct Style) 
    มีการหั่นเอกสาร (Truncation) เพื่อให้แน่ใจว่า 5090 จะประมวลผลได้ไวและแม่นยำ
    """
    # 🚨 ตัดเนื้อหาเอกสารเหลือ 2,500 ตัวอักษรต่อไฟล์ เพื่อคุม Token รวมไม่ให้เกิน 8k-10k
    context_list = []
    for doc in retrieved_docs:
        truncated_content = doc['content'][:2500] 
        context_list.append(f"เอกสาร: {doc['filename']}\nเนื้อหา: {truncated_content}")
    
    context = "\n\n---\n\n".join(context_list)
    choice_lines = "\n".join(f"{i+1}. {c}" for i, c in enumerate(choices))
    
    # ใช้ Prompt Format แบบ Instruct ที่ Typhoon เข้าใจได้ดีที่สุด
    return f"""คุณคือผู้ช่วยอัจฉริยะที่ตอบคำถามได้อย่างถูกต้องและแม่นยำ โดยอ้างอิงจากเอกสารที่กำหนดให้เท่านั้น

    [เอกสารอ้างอิง]
    {context}

    [คำถาม]
    {question}

    [ตัวเลือก]
    {choice_lines}

    [กฎการตอบ]
    1. วิเคราะห์ข้อมูลจากเอกสารอ้างอิงเท่านั้น ห้ามใช้ความรู้ภายนอก
    2. หากไม่พบคำตอบในเอกสาร ให้เลือกข้อที่เป็นการบอกว่าไม่มีข้อมูล
    3. ตอบเป็น "ตัวเลขโดด" (1-10) ตามด้วยเลขข้อที่ถูกต้องเพียงตัวเดียวเท่านั้น

    คำตอบ: """ 

def call_ollama(prompt: str) -> str:
    """
    POST prompt ไปยัง Ollama โดยปรับจูนพารามิเตอร์สำหรับ RTX 5090
    """
    payload = {
        "model": LLM_MODEL_NAME, 
        "prompt": prompt, 
        "stream": False,
        "options": {
            "num_ctx": 24000,     # 5090 รับได้สบายมาก ตั้งเผื่อไว้สำหรับไฟล์ยาว
            "num_predict": 50,    # เราต้องการแค่ตัวเลข ไม่ต้องให้โมเดลร่ายยาว
            "temperature": 0.0,   # บังคับให้ตอบนิ่งและตรงไปตรงมาที่สุด
            "top_k": 1,
            "top_p": 0.9,
            "repeat_penalty": 1.1
        }
    }
    try:
        resp = requests.post(OLLAMA_API_URL, json=payload, timeout=120) # 5090 แรงมาก 2 นาทีก็เกินพอ
        resp.raise_for_status()
        return resp.json().get("response", "")
    except Exception as e:
        print(f"  ⚠️ Ollama API Error: {e}")
        return ""


def extract_final_answer(raw: str) -> str:
    """
    ดึงคำตอบจาก Typhoon-S โดยค้นหาตัวเลขโดดที่โมเดลตอบกลับมา
    """
    if not raw: return ""
    
    # ทำความสะอาดข้อความ เผื่อโมเดลตอบ "คำตอบคือข้อ 3" หรือ "3"
    clean = raw.strip()
    
    # ค้นหาคำว่า "คำตอบ:" ก่อน
    m = re.search(r"คำตอบ\s*:\s*(\d+)", clean)
    if m:
        return m.group(1).strip()
    
    # ถ้าไม่เจอคำว่า "คำตอบ:" ให้ค้นหาตัวเลขตัวแรกที่ปรากฏในข้อความ
    digits = re.findall(r'\d+', clean)
    if digits:
        return digits[0] # มักจะเป็นตัวเลือกที่โมเดลเลือก
        
    return clean

def append_trace(record: dict):
    with open(TRACE_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def clear_trace():
    if os.path.exists(TRACE_LOG_PATH):
        with open(TRACE_LOG_PATH, "w", encoding="utf-8"):
            pass

print("✅ Helper functions for Typhoon-S (RTX 5090 Optimized) Ready.")

✅ Helper functions for Typhoon-S (RTX 5090 Optimized) Ready.


## Cell 7 — Steps 4–6: Main Pipeline Loop

In [66]:
# ============================================================
#  Steps 4-6 — Retrieval → LLM → Parse → Fill submission
# ============================================================

# Work on a copy of the template; iterate by id order from sample_submission
df_result = df_submission.copy()
df_result["answer"] = df_result["answer"].astype(object) 
q_lookup = df_questions.set_index("id")

clear_trace()

for idx, row in df_result.iterrows():
    q_id = int(row["id"])

    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(df_result)}] ID={q_id}")

    try:
        q_row    = q_lookup.loc[q_id]
        question = str(q_row["question"])
        choices  = [str(q_row[c]) for c in CHOICE_COLS]

        print(f"  ❓ {question[:80]}{'...' if len(question) > 80 else ''}")

        # --- Step 4: Retrieve + Trace ---
        retrieved           = retrieve_top_k(question)
        retrieved_filenames = [r["filename"] for r in retrieved]
        print(f"  📄 Retrieved: {retrieved_filenames}")

        append_trace({
            "question_id":     q_id,
            "question":        question,
            "retrieved_files": retrieved_filenames
        })

        # --- Step 5: LLM Generation ---
        prompt   = build_prompt(question, choices, retrieved)
        raw_resp = call_ollama(prompt)

        # --- Step 6: Parse ---
        think_text = extract_think(raw_resp)
        raw_answer = extract_final_answer(raw_resp)

        match_index = None
        is_fallback = False  # <--- Flag to track if we hit a fallback
        clean_raw = str(raw_answer).strip().lower()

        # 1. Text Matching: Try to find which exact choice text the LLM mentioned
        for i, choice_text in enumerate(choices):
            clean_choice = str(choice_text).strip().lower()
            if clean_choice and clean_choice in clean_raw:
                match_index = i + 1  # Add 1 because choices are 1-10
                break
        
        # 2. Number Fallback: Look for standalone numbers (1-10) in the LLM's answer
        if match_index is None:
            numbers = re.findall(r'\b([1-9]|10)\b', clean_raw) 
            if numbers:
                match_index = int(numbers[0])
            else:
                match_index = 1 # Absolute fallback to prevent crashes
                is_fallback = True # <--- Set the flag to True

        # --- Explicit Print Logging ---
        if is_fallback:
            # Prints a loud warning if the parser couldn't find a valid number/text
            print(f"  ⚠️ FALLBACK TRIGGERED (Defaulting to 1). Raw: '{str(raw_answer).strip()[:40]}...'")
        else:
            # Normal print for successful parsing
            print(f"  🤖 Raw Answer: {str(raw_answer).strip()[:40]}... -> Mapped to: {match_index}")

        # ---> NEW: Adding raw_response to the trace log!
        append_trace({
            "question_id":  q_id,
            "raw_response": raw_resp,
            "final_answer": match_index,
            "is_fallback":  is_fallback 
        })

        # Write the NUMBER into the submission DataFrame
        df_result.at[idx, "answer"] = int(match_index)

    except Exception as e:
        err_msg = str(e)
        print(f"  ❌ ERROR: {err_msg}")
        append_trace({"question_id": q_id, "error": err_msg})
        # Row keeps the original template value on error

# --- Save timestamped submission CSV ---
timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(SUBMISSION_DIR, f"submission_{timestamp}.csv")
df_result.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\n✅ Done! Submission saved → {output_path}")
display(df_result)


[1/100] ID=1
  ❓ Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  📄 Retrieved: ['products/WK-SW-001_wongkhojon_watch_s3_ultra.md', 'products/WK-SW-002_wongkhojon_watch_s3_pro.md', 'products/WK-SW-003_wongkhojon_watch_s3.md', 'products/WK-SW-004_wongkhojon_watch_s3_se.md']
  ⚠️ FALLBACK TRIGGERED (Defaulting to 1). Raw: '...'

[2/100] ID=2
  ❓ ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ
  📄 Retrieved: ['products/JC-CS-006_judchuam_saifah_pen_gen2.md', 'products/SF-TB-002_saifah_tab_s9.md', 'products/SF-TB-005_saifah_tab_mini_7.md', 'products/SF-TB-007_saifah_tab_draw_pro.md']
  ⚠️ FALLBACK TRIGGERED (Defaulting to 1). Raw: '...'

[3/100] ID=3
  ❓ หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ
  📄 Retrieved: ['products/KS-HP-001_kluensiang_headpro_x1.md', 'products/KS-HP-002_kluensiang_headpro_x1_se.md', 'products/KS-HP-003_kluensiang_headon_700.md', 'products/KS-HP-005_kluensiang_headon_300.md']
  ⚠️ FALLBACK TRIGGERED (Defaulting to 1). Raw: '...'

[4/100] ID=4
  ❓ อยากเอามือถือเครื่องเก่ามาเทิร์น เปล

KeyboardInterrupt: 

## Cell 8 — Sanity Check

In [51]:
# ============================================================
#  Sanity Check — submission stats & trace entries
# ============================================================

print(f"Total rows         : {len(df_result)}")
print(f"Answered (non-null): {df_result['answer'].notna().sum()}")
print(f"Still template val : {(df_result['answer'] == df_submission['answer']).sum()}")
print()
display(df_result)

with open(TRACE_LOG_PATH, "r", encoding="utf-8") as f:
    trace_lines = f.readlines()
print(f"\nTrace log entries: {len(trace_lines)}")

Total rows         : 100
Answered (non-null): 100
Still template val : 37



,id,answer
0,1,5
1,2,7
2,3,2
3,4,1
4,5,1
...,...,...
95,96,5
96,97,2
97,98,2
98,99,1



Trace log entries: 974
